In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from torch.nn import *

torch.set_default_tensor_type(torch.cuda.FloatTensor)


class F(Module):
    def __init__(s, f, a=None):
        super().__init__()
        s.f = f
        s.a = a

    def forward(s, x):
        return s.f(x) if s.a is None else s.f(x, s.a)


class Enc(Sequential):
    def forward(s, x):
        while x.size(2) > 1:
            x = super().forward(x)
        return x[..., 0]


class EncDec(Sequential):
    def __init__(s, *a, n=0):
        super().__init__(*a)
        s.n = n
    def forward(s, x):
        Y = []
        for i in range(int(np.log2(x.size(2)))-s.n):
            x = s[0](x)
            y = s[1](x)
            for i in range(i + 1):
                y = s[2](y)
            Y.append(y)
        return torch.cat(Y)


class Add(Sequential):
    def forward(s, x):
        return sum((s(x) for s in s[1:]), s[0](x))


class Res(Sequential):
    def forward(s, x):
        return super().forward(x) + x


class Rep(Sequential):
    def __init__(s, n, *a):
        super().__init__(*a)
        s.n = n

    def forward(s, x):
        for _ in range(s.n):
            x = super().forward(x)
        return x


class Mean(Add):
    def forward(s, x):
        return super().forward(x) / len(s)


class Log:
    a = None
    n = 0
    def __init__(s, m=10):
        s.m = m
    def __call__(s, *x):
        if s.a is None:
            s.a = [np.empty((2000,1-hasattr(x,'__iter__')or len(x)), 'f') for x in x]
        for a, x in zip(s.a, x):
            a[s.n] = x
        s.n += 1
        if not s.n % s.m:
            for i, a in enumerate(s.a, 1):
                plt.subplot(len(s.a), 1, i).plot(a[:s.n])
            clear_output(1)
            plt.show()

In [ ]:
a = torch.tensor(np.memmap("inews.7622", "h")).unfold(0, 32, 1)
k = 4
K = 256
m = Sequential(
    Embedding(7622, K),
    F(lambda x: x.transpose(1, 2)),
    EncDec(
        Sequential(
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                    Conv1d(K, K, 1, bias=0),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            MaxPool1d(2),
        ),
        Sequential(
            Add(
                Sequential(
                    Conv1d(K, K, 1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                    Conv1d(K, k, 1, bias=0),
                ),
                Conv1d(K, k, 1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(k, K, 1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                    Conv1d(K, K, 1, bias=0),
                ),
                Conv1d(k, K, 1),
            ),
        ),
        Sequential(
            Add(
                Sequential(
                    ConvTranspose1d(K, K, 3, 2, 1, 1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                ConvTranspose1d(K, K, 3, 2, 1, 1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
        ),
        n=2,
    ),
    Add(
        Sequential(
            Conv1d(K, K, 1, bias=0),
            BatchNorm1d(K, track_running_stats=0),
            SiLU(1),
            Conv1d(K, 7622, 1, bias=0),
        ),
        Conv1d(K, 7622, 1),
    )
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.0005, 500000)
log = Log()
for t in range(O.total_steps // 500):
    for t in range(500):
        y = a[torch.randint(len(a), (4,))].long()
        x = m(y).unflatten(0, (-1, len(y)))
        t = functional.cross_entropy(x.transpose(1, 2), y.expand(len(x), -1, -1))
        t.backward()
        o.step()
        o.zero_grad()
        O.step()
    log(t.item(), ((x.argmax(2)==y).sum((1, 2))/y.numel()).cpu())

In [ ]:
a = torch.tensor(np.memmap("inews.7622", "h")).unfold(0, 32, 1)
k = 4
K = 256
m = Sequential(
    Embedding(7622, K),
    F(lambda x: x.transpose(1, 2)),
    EncDec(
        Sequential(
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                    Conv1d(K, K, 1, bias=0),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            MaxPool1d(2),
        ),
        Sequential(
            Add(
                Sequential(
                    Conv1d(K, K, 1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                    Conv1d(K, k, 1, bias=0),
                ),
                Conv1d(K, k, 1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(k, K, 1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                    Conv1d(K, K, 1, bias=0),
                ),
                Conv1d(k, K, 1),
            ),
        ),
        Sequential(
            Add(
                Sequential(
                    ConvTranspose1d(K, K, 3, 2, 1, 1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                ConvTranspose1d(K, K, 3, 2, 1, 1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
            Add(
                Sequential(
                    Conv1d(K, K, 3, padding=1, bias=0),
                    BatchNorm1d(K, track_running_stats=0),
                    SiLU(1),
                ),
                Conv1d(K, K, 3, padding=1, bias=0),
            ),
        ),
        n=2,
    ),
    Add(
        Sequential(
            Conv1d(K, K, 1, bias=0),
            BatchNorm1d(K, track_running_stats=0),
            SiLU(1),
            Conv1d(K, 7622, 1, bias=0),
        ),
        Conv1d(K, 7622, 1),
    )
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.0002, 1000000)
log = Log()
for t in range(O.total_steps // 500):
    for t in range(500):
        y = a[torch.randint(len(a), (4,))].long()
        x = m(y).unflatten(0, (-1, len(y)))
        t = functional.cross_entropy(x.transpose(1, 2), y.expand(len(x), -1, -1))
        t.backward()
        o.step()
        o.zero_grad()
        O.step()
    log(t.item(), ((x.argmax(2)==y).sum((1, 2))/y.numel()).cpu())